# Part 3 - Machine Learning: predicting no-shows
**Goal:** flag risky bookings *before* the appointment day.

**Rules followed (from the brief):** target = `no_show`; features = `age, distance_km, fee_inr, lead_days`;
80/20 split with `random_state=42`; `StandardScaler`; judge models mainly by **Recall**
(only ~22% of bookings are no-shows, so plain accuracy is misleading);
the **test set is used for reporting only** - model *selection* and *tuning* use cross-validation on the training set.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO-NAME.git"   # only used in Colab if files are missing
if not os.path.exists("clean_appointments.csv"):
    subprocess.run(["git", "clone", REPO_URL, "project"], check=True)
    os.chdir("project")
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

## Data preparation

In [ ]:
df = pd.read_csv("clean_appointments.csv")
FEATURES = ["age", "distance_km", "fee_inr", "lead_days"]
TARGET = "no_show"

X, y = df[FEATURES], df[TARGET]
print("Rows:", len(df), "| no-show share: %.1f%%" % (100 * y.mean()))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

# Fit the scaler on TRAIN only, then apply to both (so no information leaks from the test set)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
def report(name, model, Xte, yte):
    """Print and return the standard metrics on a dataset."""
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    m = {
        "Accuracy": accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred, zero_division=0),
        "Recall": recall_score(yte, pred),
        "F1": f1_score(yte, pred),
        "ROC-AUC": roc_auc_score(yte, proba),
    }
    print(f"--- {name} (test set) ---")
    for k, v in m.items():
        print(f"{k:10s}: {v:.3f}")
    cm = confusion_matrix(yte, pred)
    print("Confusion matrix [[TN FP], [FN TP]]:")
    print(cm)
    return m

## Task 3.1 - Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
m_lr = report("Logistic Regression", log_reg, X_test_s, y_test)

coef = pd.Series(log_reg.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("\nCoefficients (on standardised features; + pushes towards no-show):")
print(coef.round(3))

**How to read this:** `lead_days` has the largest positive coefficient - the longer ahead a slot is booked,
the more likely a no-show. Check the Recall above: a model can have decent accuracy while still missing most no-shows.

## Task 3.2 - Decision Tree (and the overfitting check)

In [ ]:
tree = DecisionTreeClassifier(random_state=42).fit(X_train_s, y_train)
m_dt = report("Decision Tree", tree, X_test_s, y_test)

train_acc = accuracy_score(y_train, tree.predict(X_train_s))
train_rec = recall_score(y_train, tree.predict(X_train_s))
print(f"\nOverfitting check -> train accuracy {train_acc:.3f} vs test accuracy {m_dt['Accuracy']:.3f}")
print(f"                     train recall   {train_rec:.3f} vs test recall   {m_dt['Recall']:.3f}")
print("Tree depth:", tree.get_depth(), "| number of leaves:", tree.get_n_leaves())

**Overfitting:** an unrestricted tree keeps splitting until it memorises the training rows (train score near 1.0),
then does clearly worse on unseen rows. A big gap between the train and test numbers above is the sign.

## Task 3.3 - Random Forest (100 trees)

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_s, y_train)
m_rf = report("Random Forest", forest, X_test_s, y_test)
print("\nFeature importance:")
print(pd.Series(forest.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3))

## Task 3.4 - Model comparison and selection
**Selection uses 5-fold cross-validation on the TRAINING set only** (the test set stays untouched for reporting).
Rule: rank each model on CV Recall and CV ROC-AUC, take the lowest average rank (ROC-AUC breaks ties).

In [ ]:
models = {"Logistic Regression": log_reg, "Decision Tree": tree, "Random Forest": forest}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    cvres = cross_validate(model, X_train_s, y_train, cv=cv, scoring=["recall", "roc_auc"])
    test_m = {"Logistic Regression": m_lr, "Decision Tree": m_dt, "Random Forest": m_rf}[name]
    rows.append({
        "Model": name,
        "CV Recall": cvres["test_recall"].mean(),
        "CV ROC-AUC": cvres["test_roc_auc"].mean(),
        "Test Accuracy": test_m["Accuracy"],
        "Test Precision": test_m["Precision"],
        "Test Recall": test_m["Recall"],
        "Test F1": test_m["F1"],
        "Test ROC-AUC": test_m["ROC-AUC"],
    })
comparison = pd.DataFrame(rows).set_index("Model")
comparison["avg_rank"] = (comparison["CV Recall"].rank(ascending=False)
                          + comparison["CV ROC-AUC"].rank(ascending=False)) / 2
print(comparison.round(3).to_string())

best_name = comparison.sort_values(["avg_rank", "CV ROC-AUC"], ascending=[True, False]).index[0]
print("\nSelected model (best CV Recall + ROC-AUC rank):", best_name)

### GridSearchCV on the selected model
Recall is the scoring target (`refit="recall"`); ROC-AUC is recorded alongside it.
`class_weight="balanced"` is a normal hyper-parameter: it tells the model that missing a no-show costs more.

In [ ]:
param_grids = {
    "Logistic Regression": (LogisticRegression(max_iter=1000),
                            {"C": [0.01, 0.1, 1, 10, 100], "class_weight": [None, "balanced"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42),
                      {"max_depth": [2, 3, 5, 8], "min_samples_leaf": [1, 5, 10, 20],
                       "class_weight": [None, "balanced"]}),
    "Random Forest": (RandomForestClassifier(random_state=42),
                      {"n_estimators": [100, 200], "max_depth": [3, 5, 8, None],
                       "min_samples_leaf": [1, 5, 10], "class_weight": [None, "balanced"]}),
}
base_model, grid = param_grids[best_name]

search = GridSearchCV(base_model, grid, scoring={"recall": "recall", "roc_auc": "roc_auc"},
                      refit="recall", cv=cv)
search.fit(X_train_s, y_train)

print("Best parameters:", search.best_params_)
print("Best CV recall :", round(search.best_score_, 3))
print("CV ROC-AUC of that setting:", round(search.cv_results_["mean_test_roc_auc"][search.best_index_], 3))

final_model = search.best_estimator_
m_final = report(f"TUNED {best_name}", final_model, X_test_s, y_test)

## Task 3.5 - Risk score generation
**Risk score = the model's predicted probability of a no-show (0 to 1).**
Risk bands (Low / Medium / High) use the 33rd and 67th percentile of the *training* scores as cut-offs,
so the split is not influenced by the test set.

In [ ]:
train_scores = final_model.predict_proba(X_train_s)[:, 1]
low_cut, high_cut = np.percentile(train_scores, [100 / 3, 200 / 3])
print(f"Risk band cut-offs: Low < {low_cut:.3f} <= Medium < {high_cut:.3f} <= High")


def band(score):
    return "Low" if score < low_cut else ("Medium" if score < high_cut else "High")


test_out = df.loc[X_test.index, ["appointment_id", "clinic_id"] + FEATURES + [TARGET]].copy()
test_out["risk_score"] = final_model.predict_proba(X_test_s)[:, 1].round(4)
test_out["risk_band"] = test_out["risk_score"].apply(band)
test_out["predicted_no_show"] = final_model.predict(X_test_s)
test_out = test_out.sort_values("risk_score", ascending=False)
test_out.to_csv("test_risk_score.csv", index=False)
print("Saved test_risk_score.csv ->", test_out.shape)

print("\nActual no-show rate inside each risk band (test set):")
print(test_out.groupby("risk_band")[TARGET].agg(rows="count", no_show_rate="mean").round(3)
      .reindex(["Low", "Medium", "High"]))
test_out.head(10)

### Extra file for Part 4: scores for ALL bookings
Part 4 needs a risk score for follow-up-note appointments, and only some of them fall in the test set.
So we also score every row and label which split it belongs to. **Scores on `train` rows are in-sample**
(the model saw those rows while learning), so treat `test` rows as the honest ones.

In [ ]:
all_out = df[["appointment_id", "clinic_id"] + FEATURES + [TARGET]].copy()
all_out["risk_score"] = final_model.predict_proba(scaler.transform(df[FEATURES]))[:, 1].round(4)
all_out["risk_band"] = all_out["risk_score"].apply(band)
all_out["split"] = np.where(df.index.isin(X_test.index), "test", "train")
all_out.to_csv("all_risk_scores.csv", index=False)
print("Saved all_risk_scores.csv ->", all_out.shape)
print(all_out["split"].value_counts().to_dict())